# use clio_dev3 kernel

## SSEval Communication

In [48]:
from eval_service_common.eval_service import EvalService
eval_service = EvalService()
EVAL_SET = "Hooman_eval_smoke_test_t2v_1"
ASSETS_PATH = "/sensei-fs/users/hshariati/del/clineto/html/baseline_evalEndPoint_11-08-24_1"

In [49]:
run_id = eval_service.create_generation(
    eval_set=EVAL_SET,
    model_input_type="PromptAndNegativePromptInput",
    model_id="Hooman_11-01-24_T2V_filix2_model",
    created_by="hshariati",
    num_of_variations=1,
)

In [ ]:
run_id

In [50]:
from clio.prototyping_tools.aws_util import S3Handle
from botocore import UNSIGNED

s3_handle = S3Handle(
    aws_access_key_id=None,
    aws_secret_access_key=None,
    aws_session_token=None,
    unsigned=True,
)

# ensure that the S3 client is unsigned
assert s3_handle.s3.meta.config.signature_version == UNSIGNED

In [51]:
from pathlib import Path
s3_handle.upload_dir(
    dir_path=Path(ASSETS_PATH),
    bucket=EvalService.EVAL_BUCKET_NAME,
    s3_key_prefix=run_id,
)

In [53]:
from eval_service_common.api_types import AssetCreateRequest
from eval_service_common.model_input_types import PromptAndNegativePromptInput
from pathlib import Path
# Directory containing the mp4 files
directory = Path(ASSETS_PATH)

# Initialize the asset_create_list
asset_create_list = []

# Loop over all mp4 files in the directory
for idx, mp4_file in enumerate(directory.glob("*.mp4"), start=0):
    # print(idx, mp4_file)
    asset_create_list.append(
        AssetCreateRequest(
            asset_url=eval_service.get_asset_url(run_id, str(mp4_file)),
            model_input=PromptAndNegativePromptInput(
                eval_set_item_id=idx,
                eval_set_name=EVAL_SET,
                prompt="A person is playing the guitar",
                negative_prompt="cartoon"
            ),
            mime_type="video/mp4",
        )
    )

In [ ]:
asset_create_list

In [ ]:
eval_service.create_assets(run_id=run_id, assets=asset_create_list)

### Example of Converting HTML to CSV (DF), re-transcoding the video with higher compression, and coverting back to html

In [ ]:
#Takes an html and returns DFs (tested by hs on oct 11)
from bs4 import BeautifulSoup
import pandas as pd
from piat_slim import S3Handle
def html2df(url):
    s3 = S3Handle()
    def url_to_s3_bucket_key(url, bucket='clio.corp.adobe.com'):
        return bucket, url.split(bucket)[-1].lstrip('/')
    def format_prompt(prompt):
        return ' - '.join([f"[{k.strip().replace('[','').replace(']','')}] : {v.strip().rstrip(',')}" for c in prompt.split('\n') if ': ' in c for k, v in [c.split(': ', 1)]])
    root = url.split('clio.corp.adobe.com/')[-1].split('/model_image_comparison')[0]
    bucket, key = url_to_s3_bucket_key(url)
    html_path = s3.download_file(bucket=bucket, key=key)
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    data = []
    for row in soup.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) == 3:
            prompt = cells[0].get_text(strip=True)
            video1_url = cells[1].find('video')['src'] if cells[1].find('video') else None
            video2_url = cells[2].find('video')['src'] if cells[2].find('video') else None
            production_info = prompt.split('\n')[0]
            full_prompt = f"[{production_info}] - " + format_prompt(prompt)
            if video1_url and not video1_url.startswith('http'):
                video1_url = f'https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/{root}/{video1_url.lstrip("./")}'
            if video2_url and not video2_url.startswith('http'):
                video2_url = f'https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/{root}/{video2_url.lstrip("./")}'
            data.append({'prompt': full_prompt, 'video1': video1_url, 'video2': video2_url})
    return pd.DataFrame(data)
url = 'http://clio.corp.adobe.com/files/merged_content_2024-10-10-21-19-01-4106f80832_md5_c1daee86656c227b8fd63c608e732426.html'
df = html2df(url)
print(df.iloc[0].video1)
print(df.iloc[0].prompt)
df.head()

In [1]:
# same as above. This takes html and returns DF. But it works with URLs below
from bs4 import BeautifulSoup
import pandas as pd
from piat_slim import S3Handle

def html2df_hs_v1(url):
    s3 = S3Handle()

    def url_to_s3_bucket_key(url, bucket='clio.corp.adobe.com'):
        return bucket, url.split(bucket)[-1].lstrip('/')

    def format_prompt(prompt):
        return ' - '.join([f"[{k.strip().replace('[','').replace(']','')}] : {v.strip().rstrip(',')}" for c in prompt.split('\n') if ': ' in c for k, v in [c.split(': ', 1)]])

    root = url.split('clio.corp.adobe.com/')[-1].split('/model_image_comparison')[0]

    if 'kalarot-clio-data' in url:
        bucket = 'kalarot-clio-data'
        key = url.split(bucket)[-1].lstrip('/')
    else:
        bucket, key = url_to_s3_bucket_key(url)

    html_path = s3.download_file(bucket=bucket, key=key)

    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')

    data = []
    rows = soup.find_all('tr')

    # Debug: check if rows are found
    print(f"Found {len(rows)} rows in the HTML.")

    for row in rows:
        cells = row.find_all('td')

        # Debug: check if the row contains the expected number of cells
        print(f"Found {len(cells)} cells in this row.")

        # Assuming cell[0] contains the prompt and cell[1] contains the video URLs or other content
        prompt = cells[0].get_text(strip=True)
        video_urls = cells[1].find_all('video')

        video1_url = None
        video2_url = None

        # Check if the videos exist and have a 'src' attribute
        if len(video_urls) > 0 and 'src' in video_urls[0].attrs:
            video1_url = video_urls[0]['src']
        if len(video_urls) > 1 and 'src' in video_urls[1].attrs:
            video2_url = video_urls[1]['src']

        # Ensure URLs are complete and correct
        if video1_url and not video1_url.startswith('http'):
            video1_url = f'https://s3.us-west-2.amazonaws.com/{bucket}/{root}/{video1_url.lstrip("./")}'
        if video2_url and not video2_url.startswith('http'):
            video2_url = f'https://s3.us-west-2.amazonaws.com/{bucket}/{root}/{video2_url.lstrip("./")}'

        # Prepare full prompt and store data
        production_info = prompt.split('\n')[0]
        full_prompt = f"[{production_info}] - " + format_prompt(prompt)

        # Debug: check the final prompt and video URLs before adding them to data
        print(f"Full Prompt: {full_prompt}, Video1: {video1_url}, Video2: {video2_url}")

        data.append({'prompt': full_prompt, 'video1': video1_url, 'video2': video2_url})

    # Return DataFrame and also debug the number of entries
    print(f"Data contains {len(data)} rows.")
    return pd.DataFrame(data)

# Test with different URLs
# url = 'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/files/reencoded_videos_md5_ac871b9ccb29ce914e7a1b4c35f6c610.html'
# df = html2df(url)
# df.head()


In [ ]:
# function to re-encode the video. src can be both url and file path and it takes a filter ex: yadif,deshake,deflicker,minterpolate
import os
import subprocess
from piat_slim import S3Handle
import uuid

def reencode_video(src_video, output_bucket='kalarot-clio-data', input_bucket=None, compression_level=32, fps=24, filter=None, temp_folder='/tmp/'):
    s3 = S3Handle()

    # Download the video if it's not a local file
    if not os.path.isfile(src_video):
        known_buckets = ['clio.corp.adobe.com', 'kalarot-clio-data']
        bucket = input_bucket or next((b for b in known_buckets if b in src_video), None)
        key = src_video.split(bucket)[-1].lstrip('/')
        src_video = s3.download_file(bucket=bucket, key=key, destination_folder=temp_folder)

    assert os.path.isfile(src_video)

    # Generate a local output video file path
    local_output_video = f'{temp_folder}/{uuid.uuid4()}.mp4'

    # Construct the ffmpeg command
    if filter:  # If a filter is specified, add it to the ffmpeg command
        command = f'ffmpeg -y -i {src_video} -vf {filter} -c:v libx264 -crf {compression_level} -r {fps} -movflags faststart -loglevel quiet {local_output_video}'
    else:  # If no filter is specified, use the base command
        command = f'ffmpeg -y -i {src_video} -c:v libx264 -crf {compression_level} -r {fps} -movflags faststart -loglevel quiet {local_output_video}'

    # Run the ffmpeg command and check for success
    if subprocess.run(command.split()).returncode != 0:
        return False

    # Upload the re-encoded video to S3
    uploaded_key = s3.upload_file(file_path=local_output_video, folder='reencoded_videos', bucket=output_bucket, add_md5=True)

    # Return the S3 URL of the uploaded file
    return f'https://s3.us-west-2.amazonaws.com/{output_bucket}/{uploaded_key}'

# Example usage
# reencode_video(src_video='/sensei-fs/users/hshariati/del/clineto/flickering_videos/static1.mp4', filter='deflicker', output_bucket='kalarot-clio-data')
# or url:
# src_video_url= 'http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/hshariati/del/clineto/html/baseline_evalEndPoint_11-08-24_2/286e1491-8a57-4c73-be74-e6b6e0b0f70d.mp4'


In [ ]:
# function to re-encode a df (csv)
import pandas as pd
from tqdm import tqdm
from piat_slim import S3Handle

def reencode_df(df, video_columns=[],compression_level=32):
    df = df.copy()
    assert set(video_columns).issubset(df.columns)
    for index, row in tqdm(df.iterrows(), total=len(df)):
        for video_column in video_columns:
            df.at[index, video_column] = reencode_video(row[video_column],compression_level=compression_level)
    return df
video_columns = ['video2'] #Edit your df's video_columns here
dfr = reencode_df(df.sample(2), video_columns)
dfr.head()

In [28]:
# df to html uses piat_slim S3Handle fix to render text columns properly
import pandas as pd
from piat_slim import S3Handle

def df_to_html(df: pd.DataFrame, img_columns: list = [], video_columns: list = [], width: int = 512, file_name: str = 'media.html') -> str:
    df = df.copy()

    # Ensure unique column names
    df.columns = [f'{col}_{i}' if df.columns.tolist().count(col) > 1 else col for i, col in enumerate(df.columns)]

    # Apply HTML img and video embedding only to specified columns
    for col in img_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<img src="{url}" width="{width}">' if isinstance(url, str) and url.startswith('http') else url)
    for col in video_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<video width="{width}" controls autoplay muted><source src="{url}" type="video/mp4"></video>' if isinstance(url, str) and url.startswith('http') else url)

    # Convert to HTML and save
    html = df.to_html(escape=False).replace('border="1"', 'border="0"')
    with open(file_name, 'w') as f:
        f.write(html)

    # Upload to S3
    s3 = S3Handle()
    return f'https://s3.us-west-2.amazonaws.com/kalarot-clio-data/{s3.upload_file(file_name)}'

# Example usage:
# video_columns = ['video1', 'video2']
# url = df_to_html(df_comb_2, video_columns=video_columns)


In [3]:
# another df to html this doesn't use the piat_slim s3 handle and handles text columns too
import pandas as pd
import os

def df_to_html_hs_v1(df: pd.DataFrame, output_dir: str, img_columns: list = [], video_columns: list = [], width: int = 512) -> str:
    output_file_name = f'{output_dir}/output.html'
    df = df.copy()

    # Ensure unique column names
    df.columns = [f'{col}_{i}' if df.columns.tolist().count(col) > 1 else col for i, col in enumerate(df.columns)]

    # Apply HTML img and video embedding only to specified columns
    for col in img_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<img src="{url}" width="{width}">' if isinstance(url, str) and url.startswith('http') else url)
    for col in video_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda url: f'<video width="{width}" controls autoplay muted><source src="{url}" type="video/mp4"></video>' if isinstance(url, str) and url.startswith('http') else url)

    # Convert to HTML and save
    html = df.to_html(escape=False).replace('border="1"', 'border="0"')
    with open(output_file_name, 'w') as f:
        f.write(html)

    # Upload to S3
    html_url = f"s3://clio.corp.adobe.com/model_evaluation/{output_dir}/output.html"
    os.system(f"aws s3 cp {output_file_name} {html_url} --no-sign-request")
    return html_url.replace("s3://", "https://")

# Example usage:
# video_columns = ['video1', 'video2']
# url = df_to_html_hs_v1(df_baseline, output_dir='/sensei-fs/users/hshariati/hs_scripts/del', video_columns=video_columns)
# print(url)


In [ ]:
# this takes a folder of videos as input, re-encodes them, stores the results in a DF, and then renders the DF into HTML
# filters can be: yadif,deshake,deflicker,minterpolate
import os
import pandas as pd
from tqdm import tqdm

# Function to scan a folder for video files
def get_video_files_from_folder(folder_path, extensions=['.mp4', '.mkv', '.avi']):
    video_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                video_files.append(os.path.join(root, file))
    return video_files

# Main function to re-encode videos from a folder and generate HTML
def process_videos_from_folder(folder_path, output_bucket='kalarot-clio-data', compression_level=32, fps=24, filter=None, width=512, html_file_name='reencoded_videos.html'):
    video_files = get_video_files_from_folder(folder_path)
    data = []

    # Re-encode each video
    for video_file in tqdm(video_files):
        reencoded_url = reencode_video(video_file, output_bucket=output_bucket, compression_level=compression_level,fps=fps,filter=filter)
        data.append({'original_video': video_file, 'reencoded_video': reencoded_url})

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Render DataFrame as HTML with embedded videos
    html_url = df_to_html(df, video_columns=['reencoded_video'], width=width, file_name=html_file_name)

    return html_url

# Example usage
folder_path = '/sensei-fs/users/hshariati/del/clineto/flickering_videos'  # Change this to the folder containing your videos
html_url = process_videos_from_folder(folder_path, filter=None, compression_level=40)
print(f'HTML output: {html_url}')


In [ ]:
# load csv into df
import pandas as pd

# Specify the path to your CSV file
csv_file_path = 'final_hs_dfs_c2.csv'

# Load the CSV file into a DataFrame
final_df2 = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame
print(final_df2.head())

## df utilities

In [ ]:
# df utilities

# Selecting first two columns from df1 and second column from df2
combined_df = pd.concat([df_baseline.iloc[:, :2], df_ref_dec.iloc[:, 1]], axis=1)

# combined_df will now have the first two columns from df1 and the second column from df2

# this will set the columns to these names
combined_df.columns = ['New_Column1', 'New_Column2', 'New_Column3']

# Assuming 'combined_df' is your DataFrame
combined_df.columns = ['prompt', 'production', 'ref_decoder_V2']

# drop the first column
new_comb = combined_df.drop(combined_df.columns[0], axis=1)


# Assuming 'df' is your DataFrame with two columns of URLs
df_baseline = df_baseline.apply(lambda col: col.str.replace('/t2v.html', '', regex=False))

# save the df as csv
final_df2.to_csv('final_hs_dfs_c2.csv', index=False)

# load a csv into a df from disk
df_baseline = pd.read_csv('baseline_df.csv')

## other examples

In [ ]:
# this takes a folder of videos as input, re-encodes them, stores the results in a DF, and then renders the DF into HTML
# filters can be: yadif,deshake,deflicker,minterpolate
import os
import pandas as pd
from tqdm import tqdm

# Function to scan a folder for video files
def get_video_files_from_folder(folder_path, extensions=['.mp4', '.mkv', '.avi']):
    video_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                video_files.append(os.path.join(root, file))
    return video_files

# Main function to re-encode videos from a folder and generate HTML
def process_videos_from_folder(folder_path, output_bucket='kalarot-clio-data', compression_level=32, fps=24, filter=None, width=512, html_file_name='reencoded_videos.html'):
    video_files = get_video_files_from_folder(folder_path)
    data = []

    # Re-encode each video
    for video_file in tqdm(video_files):
        reencoded_url = reencode_video(video_file, output_bucket=output_bucket, compression_level=compression_level,fps=fps,filter=filter)
        data.append({'original_video': video_file, 'reencoded_video': reencoded_url})

    # Create a DataFrame
    df = pd.DataFrame(data)
    final_hs_dfs.append(df)

# Example usage
folder_path = '/sensei-fs/users/hshariati/del/clineto/flickering_videos'  # Change this to the folder containing your videos
process_videos_from_folder(folder_path, filter='minterpolate')

In [ ]:
# get video info
import json
import os
import subprocess
from piat_slim import S3Handle
def get_video_info(src_video, temp_folder='/tmp/', input_bucket=None):
    if not os.path.isfile(src_video):
        s3 = S3Handle()
        known_buckets = ['clio.corp.adobe.com', 'kalarot-clio-data']
        bucket = input_bucket or next((b for b in known_buckets if b in src_video), None)
        key = src_video.split(bucket)[-1].lstrip('/')
        src_video = s3.download_file(bucket=bucket, key=key, destination_folder=temp_folder)
    assert os.path.isfile(src_video)
    probe = f"ffprobe -v quiet -print_format json -show_format -show_streams -i {src_video}"
    result = subprocess.run(probe.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return json.loads(result.stdout.decode())

src_video_url = 'http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/hshariati/del/clineto/html/baseline_evalEndPoint_11-08-24_2/286e1491-8a57-4c73-be74-e6b6e0b0f70d.mp4'
get_video_info(src_video_url)

In [ ]:
# given S3 URL of video, download it to disk. Use: clio_dev3 kernel
import hashlib
import os
import random
import re
import string
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import re
import subprocess
from enum import Enum
import cv2
import numpy as np

import requests
from bs4 import BeautifulSoup
from s3_handle import S3Handle
s3 = S3Handle()

def html2df(url):
    name = url.split('/')[-1][:-5]
    root = url.split('clio.corp.adobe.com/')[-1].split(name)[0]
    bucket, key= url_to_s3_bucket_key(url)
    html_path= s3.download_file('temp/'+root,bucket=bucket,key=key)
    with open(html_path, 'r') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    data = []
    for i, row in enumerate(soup.find_all('tr')):
        cells = row.find_all('td')
        if len(cells) > 1:
            prompt = cells[0].get_text(strip=True)
            image_url = cells[1].find('video')['src'] if cells[1].find('video') else None
            prompt_data = split_prompt_components_v2(prompt)
            prompt_data['video'] = image_url
            data.append(prompt_data)

    df = pd.DataFrame(data)
    df['video'] = df['video'].str.replace('./', '/', regex=False)
    base ='https://s3.us-west-2.amazonaws.com/clio.corp.adobe.com/'
    df['video'] =base+root+'/'+df['video']
    return df

def download_video_from_s3(key, temp_dir='./tmp'):
    if 's3.us-west-2.amazonaws.com' in key or 's3-accelerate.amazonaws.com' in key:
        os.makedirs(temp_dir, exist_ok=True)
        local_filename = os.path.join(temp_dir, os.path.basename(key.split('?')[0]))
        response = requests.get(key, stream=True)
        if response.status_code == 200:
            with open(local_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            return local_filename
        return None
    else:
        return S3Handle().download_file(bucket='kalarot-clio-data', key=key, destination_folder=temp_dir)

## Sim Score between two videos

In [2]:
import hashlib
import os
import random
import re
import string
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import re
import subprocess
from enum import Enum
import cv2
import numpy as np

def get_video_codec(video_path: Path) -> str:
    # The ffprobe command is used to gather information about media files (video, audio, etc.)
    # -v error : This option sets the logging level to "error". Only errors are logged.
    # -select_streams v:0 : This option selects the first video stream in the file.
    # -show_entries stream=codec_name : This option shows only the codec_name entry of the selected stream.
    # -of default=noprint_wrappers=1:nokey=1 : This option sets the output format.
    cmd = f"ffprobe -v error -select_streams v:0 -show_entries stream=codec_name -of default=noprint_wrappers=1:nokey=1 {video_path}"
    result = subprocess.run(cmd, shell=True, capture_output=True)
    output = result.stdout.decode()
    return output.strip()


def calculate_ssim(video1_path: Path, video2_path: Path) -> float | None:
    # The ffmpeg command is used to calculate the Structural Similarity Index (SSIM) between two videos.
    # The SSIM index is a metric of similarity between two images or videos, with 1 being identical and 0 being completely different.
    # -i {video1_path} -i {video2_path} : These are the input files for the ffmpeg command.
    # -filter_complex '[0:v]fps=fps=30[ref];[1:v]fps=fps=30[dist];[ref][dist]ssim' : This is a complex filtergraph.
    # [0:v]fps=fps=30[ref] : This takes the video stream from the first input, sets its frame rate to 30 fps, and names it 'ref'.
    # [1:v]fps=fps=30[dist] : This takes the video stream from the second input, sets its frame rate to 30 fps, and names it 'dist'.
    # [ref][dist]ssim : This calculates the SSIM between 'ref' and 'dist'.
    # -c:v {video_codec} : This sets the codec for the output video stream.
    # -f null - : This discards the output, as we're only interested in the SSIM value printed to the console.
    video_codec = get_video_codec(video1_path)
    cmd = f"ffmpeg -i {video1_path} -i {video2_path} -filter_complex '[0:v]fps=fps=30[ref];[1:v]fps=fps=30[dist];[ref][dist]ssim' -c:v {video_codec} -f null -"
    result = subprocess.run(cmd, shell=True, capture_output=True)
    output = result.stdout.decode() if result.stdout else result.stderr.decode()
    match = re.search(r"All:(\d+\.\d+)", output)
    if match:
        ssim = float(match.group(1))
    else:
        ssim = None
    return ssim

In [ ]:
score = calculate_ssim('/sensei-fs/users/hshariati/del/clineto/del/with_new.mp4','/sensei-fs/users/hshariati/del/clineto/del/with_old.mp4')
print(score)

## Combine htmls column wise

In [ ]:
# get df baseline and remove the t2v.html from the video column
df_baseline = html2df_hs_v1('http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/siddhantg/dumps/ref_decoder/flickering/i2v/outputs/default_pipeline/i2v.html')
df_baseline = df_baseline.apply(lambda col: col.str.replace('/i2v.html', '', regex=False))

In [ ]:
# after deleting the t2v.html
# get df feature and remove the t2v.html from the video column
df_feature = html2df_hs_v1('http://clio.corp.adobe.com/model_evaluation//sensei-fs/users/siddhantg/dumps/ref_decoder/flickering/i2v/outputs/ref_decoder_v2_py/i2v.html')
df_feature = df_baseline.apply(lambda col: col.str.replace('/i2v.html', '', regex=False))

In [13]:
# Selecting first two columns from df1 and second column from df2
combined_df = pd.concat([df_baseline.iloc[:, :2], df_feature.iloc[:, 1]], axis=1)

In [14]:
# set proper columns names
combined_df.columns = ['prompt', 'baseline', 'ref_decoder_V2']

In [ ]:
# render the html with piat_slim
url = df_to_html_hs_v1(combined_df, output_dir='/sensei-fs/users/hshariati/hs_scripts/del', video_columns=combined_df.columns)
print(url)

In [ ]:
# render the html without pia slim
url = df_to_html_hs_v1(combined_df,output_dir='/sensei-fs/users/hshariati/hs_scripts/del', video_columns=combined_df.columns)
print(url)

# Sandbox

In [ ]:
df_baseline.head()

In [ ]:
df_baseline.iloc[0, 1]

In [ ]:
df_feature.iloc[0, 1]

In [ ]:
combined_df.head()

In [40]:
df_feature = df_baseline.apply(lambda col: col.str.replace('/t2v.html', '', regex=False))